In [1]:
import torch
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
from trackastra.model import Trackastra
from trackastra.tracking import graph_to_ctc, graph_to_napari_tracks, write_to_geff
from trackastra.data import example_data_bacteria
import tifffile
import napari


/opt/miniconda3/envs/trackastra/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_all_lineages_df = '/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/tracked_all_cell_data_aggregate_032626.pkl'
path_all_cell_data_df = '/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/all_cell_data_aggregate_032626.pkl'
all_lineages_df = pd.read_pickle(path_all_lineages_df)
all_cell_data_df = pd.read_pickle(path_all_cell_data_df)

experiment = 'DUMM_gitg068_baeS_100225'
base_path =f'/Volumes/mcovert/Instruments/Covert-lab-scope1/subgen_processed_data/{experiment}/hyperstacked/drift_corrected/rotated/mm_channels/subtracted'
path_to_phase_stack_dir=f'{base_path}'
path_to_labels_stack_dir =f'{base_path}/mask_kymos'
phase_list = os.listdir(path_to_phase_stack_dir)
mask_list =os.listdir(path_to_labels_stack_dir)



In [ ]:
# device = "automatic" 
# model = Trackastra.from_pretrained("general_2d_w_SAM2_features", device=device)


# for file_name in mask_list:
#         if file_name in phase_list:
#             base_name = file_name.replace('.tif', '')
#             fov_str, trench_str = base_name.split("_")
#             fov = (fov_str)
#             trench_id = (trench_str)
#             path_to_mask = f'{base_path}/mm3_segmented_subtracted_FOV_{fov}_region_{trench_id}_c_0.tif'
#             path_to_phase = f'{base_path}/subtracted_FOV_{fov}_region_{trench_id}_c_0.tif'
#             imgs=tifffile.imread(path_to_phase)
#             masks=tifffile.imread(path_to_mask)



#             track_graph, masks_tracked = model.track(imgs, masks, mode="ilp")
#             ctc_tracks, ctc_masks = graph_to_ctc(
#                   track_graph,masks_tracked,outdir=f'{base_name}_tracked_ctc"')






In [3]:
device = "automatic" 
model = Trackastra.from_pretrained("general_2d_w_SAM2_features", device=device)

path_to_mask = f'{base_path}/mm3_segmented_subtracted_FOV_018_region_1185_c_0.tif'
path_to_phase = f'{base_path}/subtracted_FOV_018_region_1185_c_0.tif'
imgs=tifffile.imread(path_to_phase)
masks=tifffile.imread(path_to_mask)



track_graph, masks_tracked = model.track(imgs, masks, mode="ilp")
ctc_tracks, ctc_masks = graph_to_ctc(
    track_graph,masks_tracked,outdir=f'018_1185_tracked_ctc"')

napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_graph)

v = napari.Viewer()
v.add_image(imgs)
v.add_labels(ctc_masks)
v.add_tracks(data=napari_tracks, graph=napari_tracks_graph)




INFO:trackastra.model.model:Loading model state from /Users/adrianjuarez/Library/Application Support/trackastra/models/general_2d_w_SAM2_features/model.pt


/Users/adrianjuarez/Library/Application Support/trackastra/models/general_2d_w_SAM2_features already downloaded, skipping.


INFO:trackastra.model.model_api:Using device mps
INFO:trackastra.model.model_api:Default batch size = 4 for model on mps.
INFO:trackastra_pretrained_feats.pretrained_features:Using model facebook/sam2.1-hiera-base-plus with mode mean_patches_exact for pretrained feature extraction.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/sam2.1-hiera-base-plus/resolve/main/sam2.1_hiera_base_plus.pt "HTTP/1.1 302 Found"
INFO:root:Loaded checkpoint sucessfully
INFO:trackastra.model.model_api:Predicting weights for candidate graph
INFO:trackastra.data.wrfeat:Extracting features from 90 frames.
INFO:trackastra.model.model_api:Building windows                      
Building windows: 100%|██████████| 87/87 [00:00<00:00, 8071.32it/s]
INFO:trackastra.model.model_api:Predicting windows with batch size 4
Computing associations: 100%|██████████| 22/22 [00:02<00:00,  9.66it/s]
INFO:trackastra.model.model_api:Running greedy tracker
INFO:trackastra.tracking.tracking:Build candidate graph with d


Candidate graph		424 nodes	722 edges
Solution graph		424 nodes	422 edges


100%|██████████| 70/70 [00:00<00:00, 249025.68it/s]


<Tracks layer 'napari_tracks' at 0x399591630>

In [ ]:
napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_graph)
# napari_tracks is an ndarray of shape (424, 4)
# napari_tracks_graph is a dict of size = 68 
# tracks_graph is a DiGraph with sizee 424 (424 nodes and 422 edges)

v = napari.Viewer()
v.add_image(imgs)
v.add_labels(ctc_masks)
v.add_tracks(data=napari_tracks, graph=napari_tracks_graph)


#imgs is an ndarraay of shape (90,382,20) that is the original phase images 
# ctc_tracks is a dataframe with shape (70,4) that containes columns label, t1, t2, parent
# ctc_masks is an ndarray that has the same dimensinos as imgs

100%|██████████| 70/70 [00:00<00:00, 15848.07it/s]


<Tracks layer 'napari_tracks' at 0x17c636e30>

In [ ]:
import numpy as np

np.savez_compressed(
    "napari_session_data_040126.npz",
    imgs=imgs,
    napari_tracks=napari_tracks,
)


import numpy as np

d = np.load("napari_session_data.npz", allow_pickle=False)
imgs = d["imgs"]
napari_tracks = d["napari_tracks"]

In [10]:
import json

with open("napari_tracks_graph.json", "w") as f:
    json.dump(napari_tracks_graph, f)

In [ ]:
import json

with open("napari_tracks_graph.json", "r") as f:
    napari_tracks_graph = json.load(f)

# JSON keys come back as strings; convert if needed
napari_tracks_graph = {int(k): int(v) for k, v in napari_tracks_graph.items()}

In [11]:
import numpy as np

np.save("napari_tracks.npy", napari_tracks)
# load later:
# napari_tracks = np.load("napari_tracks.npy")

In [12]:
import json

with open("napari_tracks_graph.json", "w") as f:
    json.dump(napari_tracks_graph, f)

# load later:
# with open("napari_tracks_graph.json") as f:
#     napari_tracks_graph = json.load(f)
# (optional) convert keys/values back to int if needed:
# napari_tracks_graph = {int(k): int(v) for k, v in napari_tracks_graph.items()}

In [13]:
np.save("imgs.npy", imgs)
# or smaller on disk:
# np.savez_compressed("imgs.npz", imgs=imgs)

# load:
# imgs = np.load("imgs.npy")
# or: imgs = np.load("imgs.npz")["imgs"]

In [16]:
np.save("ctc_masks.npy", ctc_masks)
